Step 3 Parse impression logs

In [5]:
from pathlib import Path
import pandas as pd
Data_Dir = Path("../data/raw")
train_behaviors_path = Data_Dir / "MINDsmall_train" / "behaviors.tsv"
train_news_path = Data_Dir / "MINDsmall_train" / "news.tsv"
dev_behaviors_path = Data_Dir / "MINDsmall_dev" / "behaviors.tsv"
dev_news_path = Data_Dir / "MINDsmall_dev" / "news.tsv"

In [6]:
behaviors_cols=[
 "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

news_cols = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]

train_behaviors = pd.read_csv(train_behaviors_path, sep = '\t', names=behaviors_cols, header=None)
dev_behaviors = pd.read_csv(dev_behaviors_path, sep='\t', names=behaviors_cols, header=None)
train_news = pd.read_csv(train_news_path, sep="\t", names=news_cols, header=None)
dev_news = pd.read_csv(dev_news_path, sep='\t', names=news_cols, header=None)

print("train_behaviors:", train_behaviors.shape)
print("dev_behaviors:", dev_behaviors.shape)
print("train_news:", train_news.shape)
print("dev_news:", dev_news.shape)

train_behaviors: (156965, 5)
dev_behaviors: (73152, 5)
train_news: (51282, 8)
dev_news: (42416, 8)


I converted the raw time column into pandas datetime format so that I could sort user events chronologically and support time-based splitting. I used errors="coerce" to safely handle malformed timestamps by converting them into missing datetime values, which can then be inspected or filtered.

In [7]:
train_behaviors ['time'] = pd.to_datetime(train_behaviors['time'],errors="coerce")
dev_behaviors ['time'] = pd.to_datetime(train_behaviors['time'], errors="coerce")

In [8]:
def parse_impressions(behaviors_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for row in behaviors_df.itertuples(index=False):
        impression_id = row.impression_id
        user_id = row.user_id
        time = row.time
        impressions = row.impressions

        if pd.isna(impressions):
            continue

        impressions_items = impressions.split()

        for item in impressions_items:
            try:
                item_id, click = item.rsplit("-", 1)
            except ValueError:
                continue

            records.append({
                "user_id": user_id,
                "item_id": item_id,
                "click": int(click),
                "impression_id": impression_id,
                "time": time,
                "source": "impression"  # 最小改动：这里不要写 impressions
            })

    interactions_df = pd.DataFrame(records)
    return interactions_df


# Parse train and dev impression logs
train_impressions = parse_impressions(train_behaviors)
dev_impressions = parse_impressions(dev_behaviors)

print("\ntrain_impressions:", train_impressions.shape)
print("dev_impressions:", dev_impressions.shape)

print("\nTrain parsed impressions preview:")
display(train_impressions.head())

print("\nDev parsed impressions preview:")
display(dev_impressions.head())


train_impressions: (5843444, 6)
dev_impressions: (2740998, 6)

Train parsed impressions preview:


,user_id,item_id,click,impression_id,time,source
0,U13740,N55689,1,1,2019-11-11 09:05:58,impression
1,U13740,N35729,0,1,2019-11-11 09:05:58,impression
2,U91836,N20678,0,2,2019-11-12 18:11:30,impression
3,U91836,N39317,0,2,2019-11-12 18:11:30,impression
4,U91836,N58114,0,2,2019-11-12 18:11:30,impression



Dev parsed impressions preview:


,user_id,item_id,click,impression_id,time,source
0,U80234,N28682,0,1,2019-11-11 09:05:58,impression
1,U80234,N48740,0,1,2019-11-11 09:05:58,impression
2,U80234,N31958,1,1,2019-11-11 09:05:58,impression
3,U80234,N34130,0,1,2019-11-11 09:05:58,impression
4,U80234,N6916,0,1,2019-11-11 09:05:58,impression
